In [ ]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sb

import statsmodels.formula.api as smf

In [ ]:
DATA_DIR = '../data/processed'

df_final = pd.read_csv(f'{DATA_DIR}/df_final.csv')

In [ ]:
print(df_final.info())

In [ ]:
tar = ['stroke','cancer','heart_attack','diabetes']

pd.DataFrame({col: df_final[col].value_counts()/ len(df_final)*100 for col in tar}).T

In [ ]:
cat_tot = ['depr','appetite','concen','fatigue','phactiv',

       'drug_hbp','drug_stom','drug_sleep','drug_inf','drug_oth','alcohol','gender','astar']

num_tot = ['bmi','maxgrip','age','chol','crp','trigl','cyst_c','hdl','thb','a1c']

X_cols =  cat_tot + num_tot

y_cols = ['diabetes','stroke','heart_attack','cancer']

X = df_final[X_cols]

y = df_final[y_cols]

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

sp = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=1234)

for train_index, temp_index in sp.split(X, y):

    X_train_ub, X_temp = X.iloc[train_index], X.iloc[temp_index]

    y_train_ub, y_temp = y.iloc[train_index], y.iloc[temp_index]

sp_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=1234)

for test_index, val_index in sp_val.split(X_temp, y_temp):

    X_test, X_val = X_temp.iloc[test_index], X_temp.iloc[val_index]

    y_test, y_val = y_temp.iloc[test_index], y_temp.iloc[val_index]

In [ ]:
from collections import Counter

print(Counter(y_train_ub['diabetes']))

print(Counter(y_train_ub['stroke']))

print(Counter(y_train_ub['heart_attack']))

print(Counter(y_train_ub['cancer']))

print(pd.DataFrame({ col : y_train_ub[col].value_counts()/len(y_train_ub) *100 for col in tar}).T)

print(Counter(y_test['diabetes']))

print(Counter(y_test['stroke']))

print(Counter(y_test['heart_attack']))

print(Counter(y_test['cancer']))

print(pd.DataFrame({ col : y_test[col].value_counts()/len(y_test) *100 for col in tar}).T)

print(Counter(y_val['diabetes']))

print(Counter(y_val['stroke']))

print(Counter(y_val['heart_attack']))

print(Counter(y_val['cancer']))

print(pd.DataFrame({ col : y_val[col].value_counts()/len(y_val) *100 for col in tar}).T)

In [ ]:
from imblearn.combine import SMOTEENN

from imblearn.over_sampling import SMOTE

from imblearn.under_sampling import EditedNearestNeighbours

sme = SMOTEENN(random_state=1234,

               sampling_strategy=0.5

               )

X_res_stroke, y_res_stroke = sme.fit_resample(X_train_ub, y_train_ub['stroke'])

y_res_stroke.name = 'stk'

X_res_diab, y_res_diab = sme.fit_resample(X_train_ub, y_train_ub['diabetes'])

y_res_diab.name = 'diab'

X_res_ha, y_res_ha = sme.fit_resample(X_train_ub, y_train_ub['heart_attack'])

y_res_ha.name = 'ha'

X_res_can, y_res_can = sme.fit_resample(X_train_ub, y_train_ub['cancer'])

y_res_can.name = 'canc'

In [ ]:
print(Counter(y_res_diab))

print(Counter(y_train_ub['diabetes']))

print(Counter(y_res_stroke))

print(Counter(y_train_ub['stroke']))

print(Counter(y_res_can))

print(Counter(y_train_ub['cancer']))

print(Counter(y_res_ha))

print(Counter(y_train_ub['heart_attack']))

In [ ]:
def compute_scumble(y_df: pd.DataFrame) -> float:

    Y = y_df.values

    n_samples, n_labels = Y.shape

    label_freq = Y.sum(axis=0)

    max_freq = label_freq.max()

    IRLbl = max_freq / label_freq

    scumble_values = []

    for i in range(n_samples):

        active = np.where(Y[i] == 1)[0]

        if len(active) == 0:

            scumble_values.append(0)

            continue

        IR_active = IRLbl[active]

        mean_IR = IR_active.mean()

        geom_IR = np.prod(IR_active) ** (1 / len(active))

        sc_i = 1 - (geom_IR / mean_IR)

        scumble_values.append(sc_i)

    return float(np.mean(scumble_values))

sc = compute_scumble(y_train_ub).__round__(5)

print("SCUMBLE =", sc)

if sc < 0.10:

    print("Usa MLSMOTE standard")

else:

    print("SCUMBLE alto: meglio Borderline-MLSMOTE")

In [ ]:
label_freq = y_train_ub.sum()

max_freq = label_freq.max()

IRLbl = max_freq / label_freq

print(IRLbl)

best_sample = None

best_IR = np.inf

import sys

import mlsmote

X_sub, y_sub = mlsmote.get_minority_instace(X_train_ub, y_train_ub)

minority_idx = set(mlsmote.get_index(y_train_ub))

maj_mask = ~ y_train_ub.index.isin(minority_idx)

X_maj, y_maj = X_train_ub.loc[maj_mask], y_train_ub.loc[maj_mask]

n_sample = [2000,2500,2750,2900,3500]

for i in n_sample:

    X_sub_aug, y_sub_aug = mlsmote.MLSMOTE(X_sub, y_sub,

                                           n_sample=i,

                                           random_state=1234,

                                           )

    X_bal = pd.concat([X_maj.reset_index(drop=True), X_sub_aug], axis=0).reset_index(drop=True)

    y_bal = pd.concat([y_maj.reset_index(drop=True), y_sub_aug], axis=0).reset_index(drop=True)

    label_freq = y_bal.sum()

    IR_mean = (label_freq.max() / label_freq).mean()

    print(f"n_sample={i}, IRLbl medio={IR_mean:.3f}")

    if IR_mean < best_IR:

        best_IR = IR_mean

        best_sample = i

print(f'The best number of sample is {best_sample}')

In [ ]:
X_sub_aug, y_sub_aug = mlsmote.MLSMOTE(X_sub, y_sub,

                                       n_sample=best_sample,

                                       random_state=1234)

X_bal = pd.concat([X_maj.reset_index(drop=True), X_sub_aug], axis=0).reset_index(drop=True)

y_bal = pd.concat([y_maj.reset_index(drop=True), y_sub_aug], axis=0).reset_index(drop=True)

print("\nTrain proportions AFTER OVERSAMPLING:")

print(y_bal.mean() * 100)

In [ ]:
print("Train proportions AFTER recombination:\n", y_bal.mean()*100)

print(pd.DataFrame({ col : y_bal[col].value_counts()/len(y_bal) *100 for col in tar}).T)

print("\nTrain proportions AFTER SMOTEENN (per label):")

for col, y_res in zip(tar, [y_res_stroke, y_res_diab, y_res_ha, y_res_can]):

    counts = y_res.value_counts()/len(y_res)*100

    print(f"  {col}: 0: {counts[0]:.2f}   1: {counts[1]:.2f}  N={len(y_res)}")

print("Train proportions BEFORE recombination:\n", y_train_ub.mean()*100)

print(pd.DataFrame({ col : y_train_ub[col].value_counts()/len(y_train_ub) *100 for col in tar}).T)

In [ ]:
label_freq = y_train_ub.sum()

max_freq = label_freq.max()

IRLbl_pre = max_freq / label_freq

scumble_pre = compute_scumble(y_train_ub)

print(" PRE OVERSAMPLING")

print("IRLbl:\n", IRLbl_pre)

print("SCUMBLE =", round(scumble_pre, 5))

smoteenn_labels = {

    'stroke':      y_res_stroke,

    'diabetes':    y_res_diab,

    'heart_attack': y_res_ha,

    'cancer':      y_res_can,

}

min_len = min(len(s) for s in smoteenn_labels.values())

y_smoteenn_proxy = pd.DataFrame(

    {col: s.values[:min_len] for col, s in smoteenn_labels.items()}

)

label_freq_sme = y_smoteenn_proxy.sum()

IRLbl_sme      = label_freq_sme.max() / label_freq_sme

scumble_sme    = compute_scumble(y_smoteenn_proxy)

print("\n=== POST SMOTEENN (proxy) ===")

print(f"N samples (min tra label): {min_len}")

print("IRLbl:\n", IRLbl_sme.round(3))

print("SCUMBLE =", round(scumble_sme, 5))

label_freq_post = y_bal.sum()

max_freq_post = label_freq_post.max()

IRLbl_post = max_freq_post / label_freq_post

scumble_post = compute_scumble(pd.DataFrame(y_bal))

print("POST MLSMOTE")

print("IRLbl:\n", IRLbl_post)

print("SCUMBLE =", round(scumble_post, 5))

In [ ]:
def imbalance_rate(y: pd.DataFrame) -> pd.Series:

    return y.apply(lambda col: (col == 0).sum() / (col == 1).sum())

IR_pre = imbalance_rate(y_train_ub)

scumble_pre = compute_scumble(y_train_ub)

print ("PRE OVERSAMPLING")

print(f"N samples: {len(y_train_ub)}")

print("IR:\n", IR_pre.round(3))

print("SCUMBLE =", round(scumble_pre, 5))

smoteenn_labels = {

    'stroke':       y_res_stroke,

    'diabetes':     y_res_diab,

    'heart_attack': y_res_ha,

    'cancer':       y_res_can,

}

min_len = min(len(s) for s in smoteenn_labels.values())

y_smoteenn_proxy = pd.DataFrame(

    {col: s.values[:min_len] for col, s in smoteenn_labels.items()}

)

IR_sme     = imbalance_rate(y_smoteenn_proxy)

scumble_sme = compute_scumble(y_smoteenn_proxy)

print("POST SMOTEENN")

print(f"N samples (min): {min_len}")

print("IR\n", IR_sme.round(3))

print("SCUMBLE =", round(scumble_sme, 5))

IR_post     = imbalance_rate(y_bal)

scumble_post = compute_scumble(y_bal)

print("POST MLSMOTE")

print(f"N samples: {len(y_bal)}")

print("IR:\n", IR_post.round(3))

print("SCUMBLE =", round(scumble_post, 5))

In [ ]:
import pickle

with open('balanced_data.pkl', 'wb') as b:

    pickle.dump({

        'X_bal': X_bal,

        'y_bal': y_bal,

        'X_test': X_test,

        'y_test': y_test,

        'X_val': X_val,

        'y_val': y_val,

        'X_res_can': X_res_can,

        'X_res_diab': X_res_diab,

        'X_res_stroke': X_res_stroke,

        'X_res_ha': X_res_ha,

        'y_res_can': y_res_can,

        'y_res_diab': y_res_diab,

        'y_res_stroke': y_res_stroke,

        'y_res_ha': y_res_ha,

        'x_train_ub' : X_train_ub,

        'y_train_b' : y_train_ub

    }, b)